<a href="https://colab.research.google.com/github/bah862696-coder/DI-Bootcamp/blob/master/Exercises_XP_RAG_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: RAG with LangChain (Student)

## Partie 1 — Installation

Nous allons installer toutes les bibliothèques nécessaires pour construire notre système RAG. Nous utiliserons `pip` pour installer `langchain`, `langchain-community`, `langchain-huggingface`, `faiss-cpu`, `sentence-transformers`, `transformers`, `datasets`, `accelerate`, et `torch`. La commande `-q` rend l'installation silencieuse.

In [68]:
# Installation forcée et mise à jour des packages
!pip install --upgrade -q langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers transformers datasets accelerate torch

print("Installation mise à jour. Si des erreurs d'import persistent, redémarrez le runtime via 'Exécution > Redémarrer le runtime'.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 94.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2

## Partie 2 — Importations

Maintenant que les bibliothèques sont installées, nous allons importer les modules nécessaires. Une organisation propre des imports facilite la lecture et la maintenance du code.

In [88]:
import sys
from typing import List
from datasets import load_dataset
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Imports modernes LangChain 0.3+
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
    from langchain_community.vectorstores import FAISS
    from langchain_core.documents import Document
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.runnables import RunnablePassthrough
    from langchain_core.output_parsers import StrOutputParser
    print("Importations fondamentales réussies.")
except ImportError as e:
    print(f"Erreur d'importation : {e}")
    print("CONSEIL : Redémarrez le runtime (Exécution > Redémarrer le runtime) et relancez cette cellule.")

Importations fondamentales réussies.


## Partie 3 — Chargement du dataset

Nous allons charger le dataset `m-ric/huggingface_doc` depuis Hugging Face Datasets. Nous utiliserons un sous-ensemble `train[:200]` pour un traitement plus rapide dans ce notebook. Nous examinerons ensuite sa structure en affichant ses colonnes, sa taille, une ligne complète et quelques exemples de texte pour comprendre les données.

In [ ]:
# Définition du nom du dataset et du split
dataset_name = "m-ric/huggingface_doc"
split = "train[:200]" # Utilisation d'un sous-ensemble pour des raisons de performance

# Chargement du dataset
ds = load_dataset(dataset_name, split=split)

# Affichage des informations sur le dataset
print(f"Dataset chargé : {dataset_name} (split: {split})")
print(f"Colonnes disponibles : {ds.column_names}")
print(f"Nombre de lignes dans le dataset : {len(ds)}")

print("\n--- Exemple d'une ligne complète ---")
example_row = ds[0]
for key, value in example_row.items():
    print(f"{key}: {value[:200]}{'...' if len(value) > 200 else ''}")

print("\n--- Quelques exemples de texte ---")
for i in range(min(3, len(ds))):
    print(f"\nTexte {i+1}:\n{ds[i]['text'][:300]}{'...' if len(ds[i]['text']) > 300 else ''}")

### Explication de la structure du dataset

Le dataset `m-ric/huggingface_doc` contient des documents textuels liés à Hugging Face. Il est composé de deux colonnes principales :

*   `text`: Contient le contenu textuel du document.
*   `source`: Indique l'origine du document, ce qui est utile pour la traçabilité et le débogage, notamment pour savoir d'où provient l'information récupérée par le RAG.

## Partie 4 — Conversion en documents LangChain

Pour que LangChain puisse traiter nos données, nous devons transformer chaque ligne du dataset en un objet `Document`. Chaque document aura :
*   `page_content` : le texte brut du document.
*   `metadata` : un dictionnaire contenant des informations supplémentaires, ici la clé `source`.

In [14]:
from langchain_core.documents import Document
from datasets import load_dataset

# S'assurer que le dataset est chargé
if 'ds' not in locals():
    ds = load_dataset("m-ric/huggingface_doc", split="train[:200]")

documents = []
for row in ds:
    doc = Document(
        page_content=row["text"],
        metadata={"source": row["source"]}
    )
    documents.append(doc)

print(f"Nombre de documents créés : {len(documents)}")

Nombre de documents créés : 200


## Partie 5 — Découpage des documents

Le découpage est nécessaire car :
1.  **Limites de contexte** : Les modèles d'intégration (embeddings) et les LLM ont une fenêtre de contexte limitée.
2.  **Précision** : Des segments plus petits permettent de récupérer uniquement les informations pertinentes, évitant ainsi d'envoyer trop de bruit au modèle.
3.  **Coût/Vitesse** : Traiter moins de texte est plus rapide.

Nous allons tester deux configurations de `RecursiveCharacterTextSplitter` :
*   **Configuration 1** : chunk_size=500, chunk_overlap=50
*   **Configuration 2** : chunk_size=1000, chunk_overlap=100

L'**overlap** (chevauchement) permet de conserver le contexte entre les morceaux afin qu'une information coupée en deux reste compréhensible.

In [70]:
# Utilisation des classes importées avec gestion d'erreurs
splitter_1 = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs_1 = splitter_1.split_documents(documents)

splitter_2 = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs_2 = splitter_2.split_documents(documents)

def print_stats(docs, name):
    avg_len = sum(len(d.page_content) for d in docs) / len(docs)
    print(f"--- {name} ---\nChunks: {len(docs)}, Longueur moyenne: {avg_len:.2f} car.\n")

print_stats(docs_1, "Config 1 (500/50)")
print_stats(docs_2, "Config 2 (1000/100)")

--- Config 1 (500/50) ---
Chunks: 5966, Longueur moyenne: 347.31 car.

--- Config 2 (1000/100) ---
Chunks: 2731, Longueur moyenne: 764.45 car.



## Partie 6 — Création des embeddings

Les embeddings sont des représentations vectorielles du texte. Ils permettent de capturer la sémantique : deux phrases ayant un sens proche auront des vecteurs proches dans l'espace multidimensionnel.

Nous utilisons le modèle `sentence-transformers/all-MiniLM-L6-v2`, qui est léger et performant pour les tâches de recherche sémantique locale.

In [72]:
# Utilisation des classes importées globalement
try:
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    print("Embeddings initialisés.")
except (NameError, ImportError):
    from langchain_community.embeddings import HuggingFaceEmbeddings
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    print("Embeddings initialisés (import local).")

/tmp/ipykernel_14348/2485915616.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings initialisés.


## Partie 7 — Création du Vector Store (FAISS)

FAISS (Facebook AI Similarity Search) est une bibliothèque permettant une recherche efficace par similarité de vecteurs. C'est ici que nos documents découpés sont indexés.

In [101]:
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

# Création de l'index FAISS avec les documents découpés (docs_1)
if 'docs_1' in locals() and 'embeddings' in locals():
    vectorstore = FAISS.from_documents(
        docs_1,
        embeddings,
        distance_strategy=DistanceStrategy.COSINE
    )
    print("Index FAISS créé avec succès.")
else:
    print("Erreur : 'docs_1' ou 'embeddings' non trouvés.")

Index FAISS créé avec succès.


## Partie 8 & 9 — Retriever et Vérification de cohérence

Le **Retriever** est l'interface qui renvoie les documents les plus proches d'une requête. Nous allons tester l'impact du paramètre `k` (nombre de documents retournés).

*   **Précision** : Un petit `k` évite le bruit.
*   **Couverture** : Un grand `k` augmente les chances d'avoir l'information complète.

In [10]:
question_test = "How to load a dataset from the Hub?"

def test_k_values(k_list):
    for k in k_list:
        print(f"\n--- Test avec k={k} ---")
        retriever = vectorstore.as_retriever(search_kwargs={"k": k})
        results = retriever.invoke(question_test)
        for doc in results:
            print(f"- Source: {doc.metadata['source']} | {doc.page_content[:100]}...")

test_k_values([2, 4])


--- Test avec k=2 ---


NameError: name 'vectorstore' is not defined

## Partie 10 — Création du modèle de langage (LLM)

Nous utilisons `google/flan-t5-small`, un modèle léger capable de s'exécuter localement sans clé API. Nous créons un pipeline Hugging Face pour la tâche `text2text-generation`.

In [100]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain_huggingface import HuggingFacePipeline

model_id = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

# Création manuelle du pipeline en spécifiant directement le modèle
# On utilise une tâche générique supportée pour éviter le KeyError
pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100,
    temperature=0.7,
    do_sample=True
)

# Initialisation du LLM LangChain
llm = HuggingFacePipeline(pipeline=pipe)
print(f"Modèle {model_id} initialisé avec succès.")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM',

Modèle google/flan-t5-small initialisé avec succès.


## Partie 11 — Construction de la chaîne RAG

La chaîne `RetrievalQA` combine le Retriever et le LLM. Lorsqu'une question est posée :
1. Le Retriever cherche les documents pertinents.
2. Ces documents sont ajoutés au prompt du LLM comme contexte.
3. Le LLM génère une réponse basée sur ce contexte.

In [102]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

template = """Utilisez le contexte suivant pour répondre à la question :
{context}

Question : {question}

Réponse : """
prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

if 'vectorstore' in locals() and 'llm' in locals():
    retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

    modern_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    class RAGCompat:
        def invoke(self, d):
            query = d.get("query", "")
            docs = retriever.invoke(query)
            result = modern_chain.invoke(query)
            return {"result": result, "source_documents": docs}

    qa_chain = RAGCompat()
    print("Système RAG (qa_chain) initialisé avec succès.")
else:
    print("Erreur : 'vectorstore' ou 'llm' non trouvés.")

Système RAG (qa_chain) initialisé avec succès.


## Partie 12 — Questions / Réponses

Nous allons maintenant tester notre système avec une série de questions sur l'écosystème Hugging Face.

In [13]:
questions = [
    "What is Hugging Face?",
    "What is a Dataset?",
    "What is Transformers?",
    "What is a Pipeline?",
    "What is the Hub?"
]

for query in questions:
    print(f"\n{'='*50}")
    print(f"QUESTION : {query}")

    # Exécution de la chaîne
    result = qa_chain.invoke({"query": query})

    print(f"RÉPONSE : {result['result']}")
    print("\nSOURCES UTILISÉES :")
    unique_sources = set(doc.metadata['source'] for doc in result['source_documents'])
    for source in unique_sources:
        print(f"- {source}")


QUESTION : What is Hugging Face?


NameError: name 'qa_chain' is not defined

## Partie 13 — Démonstration Comparative (RAG vs Sans-RAG)

Pour illustrer l'utilité du RAG, comparons ce que répond le modèle `flan-t5-small` seul face à ce qu'il répond lorsqu'il a accès aux documents de Hugging Face.

In [20]:
q = "How can I retrieve a model from the Hugging Face Hub?"

# 1. Approche Sans-RAG (LLM uniquement)
prompt_simple = f"Answer the following question: {q}"
res_no_rag = pipe(prompt_simple)[0]['generated_text']

# 2. Approche RAG
res_rag = qa_chain.invoke({"query": q})

print(f"QUESTION : {q}")
print(f"\nRÉPONSE SANS RAG : {res_no_rag}")
print(f"\nRÉPONSE AVEC RAG : {res_rag['result']}")
print("\nSources utilisées par le RAG :")
for doc in res_rag['source_documents']:
    print(f"- {doc.metadata['source']}")

NameError: name 'pipe' is not defined

## Analyse Finale Complète

### 1. Fonctionnement du Système
Le système repose sur le principe de **Récupération Augmentée par la Génération**. Au lieu de demander au modèle de générer une réponse à partir de ses connaissances internes (souvent limitées ou datées), nous injectons dans son prompt des segments de texte (chunks) provenant d'une base de données de confiance que nous avons préalablement indexée.

### 2. Utilité des Composants
*   **Embeddings & FAISS** : Sans eux, le système ne pourrait pas comprendre le "sens" de la question. Les embeddings traduisent les mots en concepts mathématiques, et FAISS permet de trouver instantanément les documents les plus proches dans cet espace vectoriel.
*   **Segmentation (Chunking)** : Elle est cruciale car elle permet de respecter la fenêtre de contexte du modèle (FLAN-T5-Small a une limite de tokens) tout en extrayant uniquement l'information pertinente.
*   **Le paramètre K** : Il agit comme un filtre. Un `k` trop faible peut omettre des détails, tandis qu'un `k` trop élevé peut introduire du bruit sémantique qui perturbe la génération.

### 3. Avantages et Limites
*   **Avantages** : Réduction drastique des hallucinations, traçabilité des sources (citation), et coût de calcul réduit (modèle local).
*   **Limites** : Le modèle utilisé ici (`flan-t5-small`) est excellent pour la démonstration mais possède des capacités de raisonnement limitées. Pour une production réelle, un modèle plus large (comme Llama 3 ou Mistral) via `accelerate` permettrait des réponses plus fluides.

### 4. Pistes d'amélioration
*   Utilisation d'un **Reranker** pour re-classer les résultats de FAISS.
*   Mise en place d'un **système de prompt engineering** plus complexe pour forcer le modèle à ne répondre *que* si l'information est présente dans le contexte.

## Partie 13 — Analyse finale

### Fonctionnement du RAG
Le RAG (Retrieval-Augmented Generation) fonctionne en trois étapes : Indexation (transformer les docs en vecteurs), Récupération (trouver les vecteurs proches de la question) et Génération (répondre avec le contexte).

### Pourquoi le RAG ?
*   **Éviter les hallucinations** : Un LLM seul peut inventer des faits s'il n'a pas été entraîné sur des données récentes. Le RAG le force à utiliser des sources fiables.
*   **Mise à jour facile** : On peut changer le Vector Store sans ré-entraîner le modèle.

### Rôles des composants
*   **Embeddings** : Traduisent le sens du texte en coordonnées mathématiques.
*   **FAISS** : Indexe ces coordonnées pour permettre une recherche instantanée parmi des milliers de documents.
*   **Chunks & Overlap** : La découpe assure que le texte tient dans la mémoire du modèle, tandis que l'overlap maintient la continuité sémantique.
*   **Paramètre K** : Détermine la quantité d'information fournie au modèle. Trop petit (k=1), on manque d'infos ; trop grand (k=10), on risque de perdre le modèle dans du bruit.

### Limites et Améliorations
*   **Limites** : Le modèle `flan-t5-small` est très limité en raisonnement. Les chunks peuvent parfois être trop courts pour capturer une idée complexe.
*   **Améliorations** : Utiliser un modèle plus large (Mistral, Llama 3), implémenter un 'Reranker' pour affiner la sélection des documents, ou utiliser des métadonnées plus riches pour filtrer la recherche.

## 0) Setup


In [ ]:
!pip -q install -U datasets transformers sentence-transformers faiss-cpu langchain langchain-core langchain-community langchain-text-splitters langchain-huggingface

In [ ]:
from typing import List

from datasets import load_dataset
from transformers import pipeline

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

from langchain_huggingface import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA


## 1) Load dataset and convert to Documents


In [23]:
dataset_name = "m-ric/huggingface_doc"
split = "train[:200]"

ds = load_dataset(dataset_name, split=split)

documents = []
for row in ds:
    documents.append(
        Document(
            page_content=row["text"],
            metadata={"source": row["source"]}
        )
    )

print(f"Documents convertis : {len(documents)}")
print("Exemple de source :", documents[0].metadata["source"])

Documents convertis : 200
Exemple de source : huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx


## 2) Split into chunks


In [64]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splits = splitter.split_documents(documents)
print(f"Nombre de segments : {len(splits)}")

ModuleNotFoundError: No module named 'langchain_text_splitters'

## 3) Vector store + retriever (FAISS)


In [65]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=splits,
    embedding=embeddings
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Vector store FAISS et Retriever prêts.")

ModuleNotFoundError: No module named 'langchain_community'

## 4) Build the RAG chain


In [67]:
from langchain.chains import RetrievalQA

qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    return_source_documents=True
)

print("Chaîne RAG configurée.")

ModuleNotFoundError: No module named 'langchain.chains'

## 5) Demo: RAG vs no-RAG


In [39]:
q = "How can I retrieve a model from the Hugging Face Hub?"

# No-RAG (LLM only)
no_rag_prompt = (
    "Answer the question. If you are not sure, say you are not sure.\n\n"
    f"Question: {q}\n"
    "Answer:"
)
no_rag_answer = pipe(no_rag_prompt)[0]["generated_text"]

# RAG
rag_result = qa.invoke({"query": q})

print("Q:", q)
print("\nNo-RAG answer:\n", no_rag_answer)
print("\nRAG answer:\n", rag_result["result"])
print("\nSources:")
for d in rag_result["source_documents"]:
    print("-", d.metadata.get("source"))

NameError: name 'pipe' is not defined